# RouterGuard — Step 1: setup, model load, baseline load statistics

**Gate for Step 1:** the model loads with per-expert `nn.ModuleList` intact (asserted, not assumed),
the environment pins are verified, and baseline OLMoE Gini is measured on **both** calibration and
held-out eval data.

Nothing here patches the router. This notebook only establishes the baseline that Step 3 is judged
against (`gates.gini_skip_balance = 0.20`).

Run order: `00 install` → `01 bootstrap` → `02 env` → ... → `07 gate`. If a cell fails, stop; do not
skip ahead to Step 2.

In [ ]:
# 00 — install pinned stack. On Colab, run this cell, then RESTART the runtime.
# transformers must stay on 4.48.3: v5 fuses experts into 3D tensors, which breaks
# per-expert hooks and bitsandbytes expert quantization (transformers issue #1849).
#
# The huggingface-hub and datasets bounds are NOT optional: a modern `datasets`
# requires huggingface-hub>=1.31 and will upgrade it as a side effect, after which
# transformers 4.48.3 cannot even be imported. This is the single easiest way to
# lose a Colab session; env.assert_env() also checks it.
INSTALL = True

if INSTALL:
    import subprocess, sys
    pkgs = [
        "transformers==4.48.3",
        "huggingface-hub>=0.24,<1.0",
        "datasets>=2.19,<5",
        "accelerate>=0.34",
        "bitsandbytes>=0.43.3",
        "pyyaml",
        "matplotlib",
        "sentencepiece",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    print("installed:", " ".join(pkgs))
    print("RESTART THE RUNTIME before running the next cell (Colab caches old transformers).")

In [ ]:
# 00b — verify the pins survived the install (run after restarting).
import importlib.metadata as md

for pkg in ("transformers", "huggingface-hub", "datasets", "bitsandbytes", "torch"):
    print(f"  {pkg:16} {md.version(pkg)}")

import transformers, huggingface_hub
assert transformers.__version__ == "4.48.3", f"transformers is {transformers.__version__}, not 4.48.3"
assert huggingface_hub.__version__.split(".")[0] == "0", (
    f"huggingface-hub {huggingface_hub.__version__} breaks transformers 4.48.3; "
    "pip install 'huggingface-hub<1.0' 'datasets>=2.19,<5' and restart"
)

from transformers import OlmoeForCausalLM, Qwen2MoeForCausalLM  # noqa: F401
print("imports OK")

In [ ]:
# 01 — bootstrap: make the repo importable wherever it lives.
import os, sys
from pathlib import Path

ROOT = Path(os.environ.get("ROUTERGUARD_ROOT", "/content/routerguard")).expanduser()
if not (ROOT / "routerguard" / "env.py").exists():
    # Local WSL checkout or a fresh clone.
    for candidate in (Path.home() / "routerguard", Path.cwd(), Path.cwd().parent):
        if (candidate / "routerguard" / "env.py").exists():
            ROOT = candidate
            break
if not (ROOT / "routerguard" / "env.py").exists():
    raise SystemExit(
        "routerguard not found. Set ROUTERGUARD_ROOT, or: git clone <repo> " + str(ROOT)
    )
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)
print(sorted(p.name for p in ROOT.iterdir()))

In [ ]:
# 02 — environment gate. Raises if transformers is not pinned/fallback-allowed or CUDA is absent.
import torch
from routerguard import env, models, data, stats, run as rgrun

report = env.assert_env(strict=True)
print(env.env_text())
print("transformers is primary pin:", report["transformers_is_primary"])

In [ ]:
# 03 — run bookkeeping: run_id, config.resolved.yaml, git sha, env.txt, deterministic seed.
PROTOCOL = rgrun.load_protocol(ROOT / "configs" / "protocol.yaml")
RUN = rgrun.new_run("step1_load_stats", protocol=PROTOCOL)
rgrun.set_seed(RUN.seed)
print(RUN)
print("results dir:", RUN.dir)

In [ ]:
# 04 — load OLMoE-1B-7B (dev target). precision='auto' -> bf16 on A100/L4, 8-bit on T4.
# If this OOMs, pass precision='4bit' explicitly and keep that precision for every
# comparison: pruned vs unpruned must be measured at the SAME precision.
model, tokenizer, info = models.load_olmoe(precision="auto")
print(info)
print("\nMoE structure (asserted):")
for key, value in info.structure.items():
    if key != "layer_names":
        print(f"  {key}: {value}")
print("\nnotes:", info.notes)
RUN.save_json("load_info_olmoe", info.to_dict())
RUN.record_env(model, extra={"load": info.to_dict()})

# Fail loudly if the layout is the fused v5 one.
assert info.structure["num_experts_per_layer"] == 64
assert info.structure["num_moe_layers"] == 16
assert info.structure["top_k"] == 8
print("\nnorm_topk_prob =", info.structure["norm_topk_prob"], "(False => no top-k renormalization)")

In [ ]:
# 05 — deterministic splits straight from protocol.yaml: 200x1024 calib, 100x1024 held-out eval.
splits = data.build_splits(tokenizer, PROTOCOL, cache_dir=RUN.dir / "splits")
for name, split in splits.items():
    print(split.describe())
    print("  meta:", {k: split.meta[k] for k in ("seed", "token_offset", "tokenizer")})
RUN.save_json("splits_meta", {k: v.meta for k, v in splits.items()})

In [ ]:
# 06 — baseline load statistics on BOTH splits. Unbiased (unpatched) routing.
stats_by_split = {}
for name, split in splits.items():
    report = stats.collect_load_stats(model, split, batch_size=4, device="cuda")
    stats_by_split[name] = report
    print(f"[{name}] tokens={report['tokens_seen']} layers={report['n_layers']}")
    print(
        f"  gini: mean={report['gini_mean']:.4f} max={report['gini_max']:.4f} "
        f"(worst layer {report['gini_worst_layer']}) min={report['gini_min']:.4f}"
    )
    print(
        f"  max_violation: mean={report['max_violation_mean']:.4f} "
        f"max={report['max_violation_max']:.4f}"
    )
    pooled = report["pooled"]
    print(
        f"  pooled: gini={pooled['gini']:.4f} dead={pooled['dead_experts']} "
        f"cv={pooled['cv']:.4f} top1_share={pooled['top1_share']:.4f}"
    )
    print("  per-layer gini:", {k: round(v, 3) for k, v in report["per_layer_gini"].items()})

RUN.save_json("baseline_load_stats", stats_by_split)

In [ ]:
# 07 — gate check + calibration/eval transfer (how much the imbalance is a stable property).
calib, eval_ = stats_by_split["calib"], stats_by_split["eval"]
gate = PROTOCOL["gates"]["gini_skip_balance"]

print(f"gates.gini_skip_balance = {gate}")
print(f"calib gini_mean = {calib['gini_mean']:.4f} -> {'BALANCING HAS HEADROOM' if calib['gini_mean'] > gate else 'BALANCING HAS NOTHING TO FIX'}")
print(f"eval  gini_mean = {eval_['gini_mean']:.4f}")

# Per-layer KL(calib || eval): how much the load distribution is a stable property
# of the model rather than of the sample. Small KL => the imbalance transfers.
calib_counts = {int(k): torch.tensor(v) for k, v in calib["per_layer_counts"].items()}
eval_counts = {int(k): torch.tensor(v) for k, v in eval_["per_layer_counts"].items()}
per_layer_kl = {layer: stats.kl(calib_counts[layer], eval_counts[layer]) for layer in sorted(calib_counts)}
print("per-layer KL(calib || eval) [nats]:", {k: round(v, 5) for k, v in per_layer_kl.items()})
print(f"  mean KL = {sum(per_layer_kl.values()) / len(per_layer_kl):.5f}  (gates.max_kl_nats = {PROTOCOL['gates']['max_kl_nats']})")
print()
print("interpretation:")
print("  - imbalance present on calib AND eval => load is a property of the model, not the sample")
print("  - if calib gini is already below the gate, Step 3 drops balancing and pivots to redundancy-only")

summary = {
    "gate_gini_skip_balance": gate,
    "calib_gini_mean": calib["gini_mean"],
    "eval_gini_mean": eval_["gini_mean"],
    "calib_pooled_gini": calib["pooled"]["gini"],
    "eval_pooled_gini": eval_["pooled"]["gini"],
    "calib_max_violation_max": calib["max_violation_max"],
    "eval_max_violation_max": eval_["max_violation_max"],
    "per_layer_kl_calib_eval": per_layer_kl,
    "mean_kl_calib_eval": sum(per_layer_kl.values()) / len(per_layer_kl),
    "tokens": {"calib": calib["tokens_seen"], "eval": eval_["tokens_seen"]},
    "gates": rgrun.summarize_gates(
        PROTOCOL,
        {
            "gini_skip_balance": calib["gini_mean"],
            "max_kl_nats": sum(per_layer_kl.values()) / len(per_layer_kl),
        },
    ),
}
RUN.save_json("step1_summary", summary)

print()
print(rgrun.format_gates(summary["gates"]))
decision = summary["gates"]["gini_skip_balance"]["status"]
print()
print("STEP 1 GATE: baseline imbalance measured on calib AND eval ->", decision)
if decision == "proceed_with_balancing":
    print("  => Step 3 keeps load balancing in the 2x2 ablation.")
else:
    print("  => Step 3 drops balancing as a method and pivots to redundancy-only. Do not re-tune the gate.")

In [ ]:
# 08 — plots: per-layer Gini (calib vs eval) and per-expert utilization.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

layers = sorted(calib["per_layer_gini"])
axes[0].plot(layers, [calib["per_layer_gini"][k] for k in layers], "o-", label="calib (200x1024)")
axes[0].plot(layers, [eval_["per_layer_gini"][k] for k in layers], "s--", label="eval (100x1024)")
axes[0].axhline(gate, color="red", ls=":", label=f"gate {gate}")
axes[0].set_xlabel("layer")
axes[0].set_ylabel("Gini of expert load")
axes[0].set_title("Baseline router imbalance (OLMoE)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Per-expert utilization for the worst calib layer, normalized to uniform = 1.0.
from routerguard.stats import UnbiasedLoadStats
import torch

worst = calib["gini_worst_layer"]
util = None
with UnbiasedLoadStats(model) as ls:
    for batch in splits["calib"].input_ids.split(8):
        with torch.no_grad():
            model(input_ids=batch.to("cuda"), use_cache=False)
    counts = ls.per_layer_counts()
    mean = counts[worst].float().mean().clamp_min(1e-9)
    util = (counts[worst].float() / mean).cpu()

axes[1].bar(range(len(util)), util.numpy())
axes[1].axhline(1.0, color="black", lw=0.8)
axes[1].set_xlabel("expert index")
axes[1].set_ylabel("load / uniform")
axes[1].set_title(f"Layer {worst}: per-expert utilization (calib)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RUN.path("figures", "step1_baseline_load.png"), dpi=130, bbox_inches="tight")
plt.show()
print("saved:", RUN.path("figures", "step1_baseline_load.png"))

## Step 1 gate checklist

Record these in the run before moving on:

- [ ] `transformers` version is `4.48.3` (or a logged fallback) and `env.assert_env` passed
- [ ] `assert_moe_structure` passed: 16 MoE layers x 64 experts, `nn.ModuleList`, top-8
- [ ] `assert_gates_unquantized` passed (gates stayed full precision under 8/4-bit)
- [ ] calib split is 200x1024 from wikitext **train**, eval is 100x1024 from wikitext **test**
- [ ] baseline Gini recorded for calib **and** eval, with per-layer values
- [ ] gate decision written down: is `calib_gini_mean > 0.20`?

If the Gini gate fails here, the Step 3 branch is already decided: drop balancing as a method and
pivot to redundancy-only. Do not re-tune the gate after seeing the number.

**Not in this notebook:** the patched-forward no-op equivalence test (Step 2, `tests/test_equivalence.py`).
That test must pass on both OLMoE and Qwen before any balancing or pruning code is trusted.